# GPT2 TTS — Inference & Metrics

This notebook performs:
1. Loading the trained GPT2TTS checkpoint from `gpt2tts_last_state_dict.pt`
2. Inference on 50 validation texts with four sampling strategies
3. Metric computation for **CER**, **UTMOS**, and **SECS**

The final table compares sampling strategies. Lower **CER** is better; higher **UTMOS** and **SECS** are better.

## 0. Install Dependencies

In [1]:
%pip install -q -r requirements-eval.txt


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Imports and Configuration

In [2]:
from dataclasses import dataclass
from pathlib import Path
import os
import ssl
import urllib.request
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import certifi
import torch
import torch.nn as nn
import torch.nn.functional as F

os.environ.setdefault('SSL_CERT_FILE', certifi.where())
os.environ.setdefault('REQUESTS_CA_BUNDLE', certifi.where())
urllib.request.install_opener(
    urllib.request.build_opener(
        urllib.request.HTTPSHandler(context=ssl.create_default_context(cafile=certifi.where()))
    )
)

from transformers import GPT2Model, GPT2TokenizerFast
from focalcodec import FocalCodec
from tqdm.auto import tqdm

if torch.cuda.is_available():
    torch.set_float32_matmul_precision('high')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


In [3]:
# Paths and parameters
CHECKPOINT_PATH = Path('artifacts/checkpoints/gpt2tts_last_state_dict.pt')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'No checkpoint found at: {CHECKPOINT_PATH}')
DATASET_DIR = Path('dataset')
OUTPUT_DIR = Path('artifacts/inference_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLES = 50
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Model Architecture

In [ ]:
@dataclass
class GPT2TTSConfig:
    base_model: str = 'gpt2'
    codebook_size: int = 8192
    n_special_tokens: int = 2
    freeze_bottom_n_layers: int = 2
    tie_audio_weights: bool = True
    gradient_checkpointing: bool = False


class GPT2TTS(nn.Module):
    def __init__(self, cfg: GPT2TTSConfig):
        super().__init__()
        self.cfg = cfg
        vocab_audio = cfg.codebook_size + cfg.n_special_tokens
        self.cfg.bos_id = cfg.codebook_size
        self.cfg.eos_id = cfg.codebook_size + 1

        self.base = GPT2Model.from_pretrained(cfg.base_model)
        hidden = self.base.config.n_embd

        for i, block in enumerate(self.base.h):
            if i < cfg.freeze_bottom_n_layers:
                for p in block.parameters():
                    p.requires_grad_(False)

        self.audio_emb = nn.Embedding(vocab_audio, hidden)
        self.audio_head = nn.Linear(hidden, vocab_audio, bias=False)

        if cfg.tie_audio_weights:
            self.audio_head.weight = self.audio_emb.weight

    @torch.no_grad()
    def generate_audio(
        self,
        text_ids: torch.Tensor,
        max_new_tokens: int = 500,
        temperature: float = 1.0,
        top_k: int = 0,
        top_p: float = 1.0,
        do_sample: bool = True,
    ) -> torch.Tensor:
        """Autoregressively generate audio-code tokens.

        Sampling modes:
          greedy      — set do_sample=False
          temperature — set temperature, top_k=0, top_p=1.0
          top-k       — set top_k > 0
          top-p       — set top_p < 1.0  (nucleus sampling)
        """
        self.eval()
        device = text_ids.device
        text_ids = text_ids.view(-1).long().to(device)

        text_emb = self.base.wte(text_ids).unsqueeze(0)                   # (1, T, H)
        bos_emb = self.audio_emb.weight[self.cfg.bos_id].view(1, 1, -1)  # (1, 1, H)
        prefix = torch.cat([text_emb, bos_emb], dim=1)  # (1, T+1, H)
        mask = torch.ones(1, prefix.size(1), device=device, dtype=torch.long)

        out = self.base(inputs_embeds=prefix, attention_mask=mask, use_cache=True, return_dict=True)
        past = out.past_key_values
        h = out.last_hidden_state[:, -1, :]  # (1, H)

        generated = []
        for _ in range(max_new_tokens):
            logits = self.audio_head(h)  # (1, vocab)

            logits[:, self.cfg.bos_id] = float('-inf')

            if not do_sample:
                next_token = logits.argmax(dim=-1)
            else:
                logits = logits / max(float(temperature), 1e-6)

                # top-k
                if top_k > 0:
                    k = min(top_k, logits.size(-1))
                    kth_val = torch.topk(logits, k, dim=-1).values[:, -1, None]
                    logits = logits.masked_fill(logits < kth_val, float('-inf'))

                # top-p (nucleus)
                if top_p < 1.0:
                    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                    sorted_probs = F.softmax(sorted_logits, dim=-1)
                    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
                    remove = cumulative_probs > top_p
                    remove[..., 1:] = remove[..., :-1].clone()
                    remove[..., 0] = False
                    sorted_logits = sorted_logits.masked_fill(remove, float('-inf'))
                    logits = torch.full_like(logits, float('-inf')).scatter(-1, sorted_idx, sorted_logits)

                probs = F.softmax(logits, dim=-1)
                if not torch.isfinite(probs).all() or probs.sum(dim=-1).item() <= 0:
                    next_token = logits.argmax(dim=-1)
                else:
                    next_token = torch.multinomial(probs, num_samples=1).squeeze(-1)

            token_id = int(next_token.item())
            if token_id == self.cfg.eos_id:
                break
            generated.append(token_id)

            token_emb = self.audio_emb(next_token.view(1)).view(1, 1, -1)
            mask = torch.ones(1, prefix.size(1) + len(generated), device=device, dtype=torch.long)
            out = self.base(
                inputs_embeds=token_emb,
                attention_mask=mask,
                past_key_values=past,
                use_cache=True,
                return_dict=True,
            )
            past = out.past_key_values
            h = out.last_hidden_state[:, -1, :]

        return torch.tensor(generated, dtype=torch.long, device=device)

## 3. Load Checkpoint

In [5]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

model_cfg = GPT2TTSConfig(**checkpoint['model_cfg'])
model_cfg.gradient_checkpointing = False

model = GPT2TTS(model_cfg).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

CODEC_NAME = checkpoint.get('codec_name', 'lucadellalib/focalcodec_25hz')
CODEC_SR = checkpoint.get('codec_sample_rate', 16000)

total = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {total/1e6:.1f}M parameters')
print(f'Codec: {CODEC_NAME}, SR: {CODEC_SR}')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded: 130.7M parameters
Codec: lucadellalib/focalcodec_25hz, SR: 16000


## 4. Codec and Tokenizer

In [6]:
codec = FocalCodec.from_pretrained(CODEC_NAME).to(DEVICE).eval()
for p in codec.parameters():
    p.requires_grad_(False)

tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token


def tokens_to_wav(audio_tokens: torch.Tensor) -> np.ndarray:
    """FocalCodec tokens → numpy waveform"""
    with torch.no_grad():
        wav = codec.toks_to_sig(audio_tokens.unsqueeze(0).to(DEVICE))
    return wav.squeeze().cpu().numpy()


def load_wav_16k(path: str) -> np.ndarray:
    """Load a wav file, resample it to 16 kHz, and return float32 numpy audio."""
    wav, sr = sf.read(path, dtype='float32', always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=-1)
    if sr != 16000:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=16000)
    return wav


print('Codec and tokenizer are ready')

Codec and tokenizer are ready


## 5. Prepare 50 Validation Texts

In [7]:
meta = pd.read_csv(
    DATASET_DIR / 'metadata.csv',
    sep='|', header=None, names=['id', 'text', 'text_norm']
)
meta['text_norm'] = meta['text_norm'].fillna(meta['text'])
meta['filepath'] = meta['id'].apply(
    lambda i: str(DATASET_DIR / 'wavs' / f'{i}.wav')
)
meta = meta[meta['filepath'].map(lambda p: Path(p).exists())].reset_index(drop=True)

# Reproduce the same validation split as in training.
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(meta))
split_at = int(0.9 * len(meta))
val_df = meta.iloc[perm[split_at:]].reset_index(drop=True)

# Uniformly select 50 validation samples.
sample_idx = np.linspace(0, len(val_df) - 1, N_SAMPLES, dtype=int)
sample_df = val_df.iloc[sample_idx].reset_index(drop=True)

print(f'Validation set: {len(val_df)}, selected: {len(sample_df)}')
sample_df[['id', 'text_norm']]

Validation set: 1310, selected: 50


,id,text_norm
0,LJ003-0083,"Each ward was calculated to hold twenty-four, ..."
1,LJ016-0311,that any secrecy in the treatment of the conde...
2,LJ034-0127,"Brennan testified that they were standing, whi..."
3,LJ022-0027,The most difficult place in the world to get a...
4,LJ015-0169,It was at once decided at the board to make a ...
5,LJ016-0317,but believed that a public ceremony destroyed ...
6,LJ011-0194,Friends went in pursuit and traced her to Hudd...
7,LJ011-0006,"He went to the bank, and found that no stocks ..."
8,LJ016-0050,His next job was to descend outside Newgate.
9,LJ039-0149,established that they had been previously load...


## 6. Inference — 4 Sampling Methods

| Method | temperature | top_k | top_p |
|---|---|---|---|
| **greedy** | 1.0 | — | — |
| **temperature** | 0.8 | — | — |
| **top-k** | 1.0 | 50 | — |
| **top-p (nucleus)** | 1.0 | — | 0.9 |

In [8]:
SAMPLING_METHODS = {
    'greedy': dict(temperature=1.0, top_k=0, top_p=1.0, do_sample=False),
    'temperature': dict(temperature=0.8, top_k=0,  top_p=1.0, do_sample=True),
    'top_k': dict(temperature=1.0, top_k=50, top_p=1.0, do_sample=True),
    'top_p': dict(temperature=1.0, top_k=0, top_p=0.9, do_sample=True),
}

results = []

for method_name, sample_kwargs in SAMPLING_METHODS.items():
    method_dir = OUTPUT_DIR / method_name
    method_dir.mkdir(exist_ok=True)
    print(f'\n=== {method_name} ===')

    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc=method_name):
        text_ids = torch.tensor(
            tokenizer.encode(row['text_norm'], add_special_tokens=False),
            dtype=torch.long, device=DEVICE
        )

        audio_tokens = model.generate_audio(
            text_ids, max_new_tokens=500, **sample_kwargs
        )

        if audio_tokens.numel() == 0:
            print(f'  WARNING: {row["id"]} produced empty output, skipping')
            continue

        wav = tokens_to_wav(audio_tokens)
        out_path = method_dir / f'{row["id"]}.wav'
        sf.write(str(out_path), wav, CODEC_SR)

        results.append({
            'id': row['id'],
            'text': row['text_norm'],
            'ref_path': row['filepath'],
            'gen_path': str(out_path),
            'method': method_name,
            'n_tokens': audio_tokens.numel(),
        })

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / 'results_index.csv', index=False)
print(f'\nSaved {len(results_df)} files to {OUTPUT_DIR}')
results_df.groupby('method')['n_tokens'].describe()


=== greedy ===


greedy:   0%|          | 0/50 [00:00<?, ?it/s]


=== temperature ===


temperature:   0%|          | 0/50 [00:00<?, ?it/s]


=== top_k ===


top_k:   0%|          | 0/50 [00:00<?, ?it/s]


=== top_p ===


top_p:   0%|          | 0/50 [00:00<?, ?it/s]


Saved 200 files to artifacts/inference_outputs


,count,mean,std,min,25%,50%,75%,max
method,,,,,,,,
greedy,50.0,167.56,60.321303,59.0,113.25,180.0,219.0,258.0
temperature,50.0,152.52,59.799747,3.0,114.00,155.5,201.5,263.0
top_k,50.0,154.28,54.339838,60.0,108.00,154.5,197.5,248.0
top_p,50.0,157.40,57.754229,61.0,110.25,152.5,200.0,254.0


## 7a. CER — Character Error Rate

Whisper transcribes each generated audio file. We compare the transcript with the reference text using `jiwer.cer`.

**CER** is the character-level edit distance divided by the number of reference characters. Lower values mean better intelligibility.

In [9]:
import whisper
from jiwer import cer as compute_cer

# 'base' is faster; 'medium' is usually more accurate.
asr = whisper.load_model('base', device=DEVICE)
print('Whisper base model loaded')


def transcribe(wav_path: str) -> str:
    result = asr.transcribe(wav_path, language='en', fp16=(DEVICE == 'cuda'))
    return result['text'].strip().lower()


cer_rows = []
for _, row in tqdm(results_df.iterrows(), total=len(results_df), desc='CER'):
    hypothesis = transcribe(row['gen_path'])
    reference = row['text'].lower()
    score = compute_cer(reference, hypothesis)
    cer_rows.append({
        'id': row['id'],
        'method': row['method'],
        'reference': reference,
        'hypothesis': hypothesis,
        'cer': round(score, 4),
    })

cer_df = pd.DataFrame(cer_rows)
cer_df.to_csv(OUTPUT_DIR / 'cer_results.csv', index=False)

cer_summary = (
    cer_df.groupby('method')['cer']
    .agg(['mean', 'median', 'std'])
    .round(4)
    .reset_index()
)
print('\nCER by method (lower is better):')
print(cer_summary.to_string(index=False))

Whisper base model loaded


CER:   0%|          | 0/200 [00:00<?, ?it/s]


CER by method (lower is better):
     method   mean  median    std
     greedy 0.9068  0.9217 0.0624
temperature 0.9103  0.8120 0.2976
      top_k 0.7780  0.7700 0.0541
      top_p 0.9498  0.8738 0.3337


## 7b. UTMOS — Automatic MOS

UTMOS estimates speech naturalness and quality on a MOS-like 1–5 scale without a reference audio file. Higher values are better.

In [10]:
# UTMOSv2 official sarulab-speech implementation
# Documentation: https://github.com/sarulab-speech/UTMOSv2
# UTMOSv2 is more robust here than the older `utmos` package because it avoids fairseq.
import utmosv2
import utmosv2._core.model._common
import utmosv2.model.ssl as utmosv2_ssl
from huggingface_hub import hf_hub_download

class NoAutocast:
    def __enter__(self):
        pass
    def __exit__(self, *args):
        pass
    def __call__(self, *args, **kwargs):
        return self

utmosv2._core.model._common.autocast = NoAutocast

# Avoid repeated online checks for facebook/wav2vec2-base once it is cached.
wav2vec_dir = Path(
    hf_hub_download(repo_id='facebook/wav2vec2-base', filename='config.json')
).parent
if not hasattr(utmosv2_ssl, '_orig_feature_extractor_from_pretrained'):
    utmosv2_ssl._orig_feature_extractor_from_pretrained = utmosv2_ssl.AutoFeatureExtractor.from_pretrained
if not hasattr(utmosv2_ssl, '_orig_auto_model_from_pretrained'):
    utmosv2_ssl._orig_auto_model_from_pretrained = utmosv2_ssl.AutoModel.from_pretrained

def _local_wav2vec_name(name):
    return str(wav2vec_dir) if name == 'facebook/wav2vec2-base' else name

def _feature_extractor_from_pretrained(name, *args, **kwargs):
    if name == 'facebook/wav2vec2-base':
        kwargs['local_files_only'] = True
    return utmosv2_ssl._orig_feature_extractor_from_pretrained(_local_wav2vec_name(name), *args, **kwargs)

def _auto_model_from_pretrained(name, *args, **kwargs):
    if name == 'facebook/wav2vec2-base':
        kwargs['local_files_only'] = True
    return utmosv2_ssl._orig_auto_model_from_pretrained(_local_wav2vec_name(name), *args, **kwargs)

utmosv2_ssl.AutoFeatureExtractor.from_pretrained = _feature_extractor_from_pretrained
utmosv2_ssl.AutoModel.from_pretrained = _auto_model_from_pretrained

utmosv2_checkpoint = hf_hub_download(
    repo_id='sarulab-speech/UTMOSv2',
    filename='fold0_s42_best_model.pth',
)

mos_predictor = utmosv2.create_model(
    pretrained=True,
    checkpoint_path=utmosv2_checkpoint,
    device=DEVICE,
).to(DEVICE).eval()
print('UTMOSv2 loaded')


def predict_utmos(wav_path: str) -> float:
    return float(mos_predictor.predict(input_path=wav_path, device=DEVICE, verbose=False))


utmos_rows = []
for _, row in tqdm(results_df.iterrows(), total=len(results_df), desc='UTMOS'):
    score = predict_utmos(row['gen_path'])
    score = round(score, 4) if not pd.isna(score) else np.nan
    utmos_rows.append({'id': row['id'], 'method': row['method'], 'utmos': score})

utmos_df = pd.DataFrame(utmos_rows)
utmos_df.to_csv(OUTPUT_DIR / 'utmos_results.csv', index=False)

utmos_summary = (
    utmos_df.groupby('method')['utmos']
    .agg(['mean', 'median', 'std'])
    .round(4)
    .reset_index()
)
print('\nUTMOS by method (1-5, higher is better):')
print(utmos_summary.to_string(index=False))

fold0_s42_best_model.pth:   0%|          | 0.00/819M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: /home/gllekkpc/.cache/huggingface/hub/models--facebook--wav2vec2-base/snapshots/0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded checkpoint from /home/gllekkpc/.cache/huggingface/hub/models--sarulab-speech--UTMOSv2/snapshots/506474f2b33dc77c234d668cc419be1861899cad/fold0_s42_best_model.pth
UTMOSv2 loaded


UTMOS:   0%|          | 0/200 [00:00<?, ?it/s]


UTMOS by method (1-5, higher is better):
     method   mean  median    std
     greedy 1.9117  1.8820 0.4395
temperature 2.7169  2.8258 0.4213
      top_k 2.4122  2.4014 0.3922
      top_p 2.7151  2.7544 0.4013


## 7c. SECS — Speaker Embedding Cosine Similarity

SECS compares the speaker embedding of generated audio with the reference validation wav for the same utterance.
The speaker encoder is SpeechBrain ECAPA-TDNN. Higher cosine similarity means the generated voice is closer to the reference speaker identity.

In [11]:
from speechbrain.inference.speaker import EncoderClassifier

spk_encoder = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    run_opts={'device': DEVICE},
    savedir='artifacts/spkrec-ecapa',
)
print('Speaker encoder loaded')


def get_speaker_embedding(wav_path: str) -> torch.Tensor:
    """Return an L2-normalized speaker embedding with shape (1, D)."""
    wav = load_wav_16k(wav_path)
    wav_t = torch.tensor(wav).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb = spk_encoder.encode_batch(wav_t)  # (1, 1, D)
    emb = emb.squeeze(1)                        # (1, D)
    return F.normalize(emb, dim=-1)


# Cache reference embeddings.
print('Computing reference embeddings...')
ref_embs = {}
for uid, path in tqdm(
    results_df[['id', 'ref_path']].drop_duplicates('id').values,
    desc='ref embeddings'
):
    if Path(path).exists():
        ref_embs[uid] = get_speaker_embedding(path)

secs_rows = []
for _, row in tqdm(results_df.iterrows(), total=len(results_df), desc='SECS'):
    if row['id'] not in ref_embs:
        continue
    gen_emb = get_speaker_embedding(row['gen_path'])
    ref_emb = ref_embs[row['id']]
    sim = float((gen_emb * ref_emb).sum().item())
    secs_rows.append({'id': row['id'], 'method': row['method'], 'secs': round(sim, 4)})

secs_df = pd.DataFrame(secs_rows)
secs_df.to_csv(OUTPUT_DIR / 'secs_results.csv', index=False)

secs_summary = (
    secs_df.groupby('method')['secs']
    .agg(['mean', 'median', 'std'])
    .round(4)
    .reset_index()
)
print('\nSECS by method (-1 to 1, higher is better):')
print(secs_summary.to_string(index=False))

hyperparams.yaml: 0.00B [00:00, ?B/s]

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Speaker encoder loaded
Computing reference embeddings...


ref embeddings:   0%|          | 0/50 [00:00<?, ?it/s]

SECS:   0%|          | 0/200 [00:00<?, ?it/s]


SECS by method (-1 to 1, higher is better):
     method   mean  median    std
     greedy 0.0801  0.0769 0.0985
temperature 0.3749  0.3866 0.1189
      top_k 0.3326  0.3300 0.0845
      top_p 0.3443  0.3581 0.1100


## 8. Summary Table

In [12]:
summary = (
    cer_summary.rename(columns={'mean': 'CER_mean', 'median': 'CER_med', 'std': 'CER_std'})
    .merge(
        utmos_summary.rename(columns={'mean': 'UTMOS_mean', 'median': 'UTMOS_med', 'std': 'UTMOS_std'}),
        on='method'
    )
    .merge(
        secs_summary.rename(columns={'mean': 'SECS_mean', 'median': 'SECS_med', 'std': 'SECS_std'}),
        on='method'
    )
    .sort_values('CER_mean')
    .reset_index(drop=True)
)

print('=' * 65)
print('FINAL SUMMARY TABLE  (50 samples, LJ Speech validation set)')
print('CER↓ lower is better  |  UTMOS↑ and SECS↑ higher are better')
print('=' * 65)
print(summary[['method', 'CER_mean', 'UTMOS_mean', 'SECS_mean']].to_string(index=False))

summary.to_csv(OUTPUT_DIR / 'metrics_summary.csv', index=False)
print(f'\nDetailed results saved to: {OUTPUT_DIR}')
summary

FINAL SUMMARY TABLE  (50 samples, LJ Speech validation set)
CER↓ lower is better  |  UTMOS↑ and SECS↑ higher are better
     method  CER_mean  UTMOS_mean  SECS_mean
      top_k    0.7780      2.4122     0.3326
     greedy    0.9068      1.9117     0.0801
temperature    0.9103      2.7169     0.3749
      top_p    0.9498      2.7151     0.3443

Detailed results saved to: artifacts/inference_outputs


,method,CER_mean,CER_med,CER_std,UTMOS_mean,UTMOS_med,UTMOS_std,SECS_mean,SECS_med,SECS_std
0,top_k,0.7780,0.7700,0.0541,2.4122,2.4014,0.3922,0.3326,0.3300,0.0845
1,greedy,0.9068,0.9217,0.0624,1.9117,1.8820,0.4395,0.0801,0.0769,0.0985
2,temperature,0.9103,0.8120,0.2976,2.7169,2.8258,0.4213,0.3749,0.3866,0.1189
3,top_p,0.9498,0.8738,0.3337,2.7151,2.7544,0.4013,0.3443,0.3581,0.1100


## 9. Listen to Examples

In [13]:
from IPython.display import Audio, display

example_id = sample_df.iloc[0]['id']
example_text = sample_df.iloc[0]['text_norm']
print(f'ID: {example_id}')
print(f'Text: {example_text}\n')

for method in SAMPLING_METHODS:
    path = OUTPUT_DIR / method / f'{example_id}.wav'
    if path.exists():
        print(f'--- {method} ---')
        display(Audio(str(path)))

print('--- reference (original) ---')
display(Audio(sample_df.iloc[0]['filepath']))

ID: LJ003-0083
Text: Each ward was calculated to hold twenty-four, allowing each individual one foot and a half;

--- greedy ---


--- temperature ---


--- top_k ---


--- top_p ---


--- reference (original) ---
